# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, referencing entities by their `@id` fields as per the Croissant schema.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print summary metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

# Print other relevant dataset details
print(f"\nAuthors: {dataset.metadata.author}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Explore record sets, their field and column `@id`s. The exploration is always based on the Croissant `@id` identifiers.

In [ ]:
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]
print(f"Record sets found: {record_set_ids}\n")
for record_set_id in record_set_ids:
    rs_meta = None
    for rs in dataset.metadata.recordSet:
        if rs['@id'] == record_set_id:
            rs_meta = rs
            break
    print(f"Record set: {record_set_id}")
    print(f"  Name: {rs_meta.get('name', '')}")
    print(f"  Description: {rs_meta.get('description', '')}")
    field_ids = [f['@id'] for f in rs_meta.get('field', [])]
    print(f"  Fields: {field_ids}")
    # For each field, print details
    for f in rs_meta.get('field', []):
        print(f"    Field: {f['@id']}, col: {f.get('column')}, dataType: {f.get('dataType', '')}")
    print()

## 3. Data Extraction
Load each record set's data as DataFrames for exploration. All references use the `@id` notation.

In [ ]:
# Extract all record set data
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for record set {rs_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))
    print()

# For further analysis we'll select the first available record set
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Analyze numeric and categorical data. We use field and column `@id` for selection and grouping. Replace the `numeric_field_id` and `group_field_id` with the actual `@id`s identified above.

In [ ]:
# ---- Setup: Pick some field IDs by examining the fields from the overview cell ----
# For this example, let's try to automatically pick a numeric field and a grouping field:
numeric_field_id = None
group_field_id = None

if main_record_set_id:
    main_rs_meta = next(iter(filter(lambda x: x['@id'] == main_record_set_id, dataset.metadata.recordSet)), None)
    for fld in main_rs_meta.get('field', []):
        dt = fld.get('dataType', '').lower()
        if dt in ['schema:number', 'schema:float', 'schema:integer'] and not numeric_field_id:
            # Use the field['@id'] as column name
            numeric_field_id = fld['@id']
        # Pick a groupable field (categorical field)
        if dt in ['schema:text', 'schema:string'] and not group_field_id:
            group_field_id = fld['@id']
        if numeric_field_id and group_field_id:
            break

if not main_record_set_id or not numeric_field_id:
    print("No suitable numeric field found for EDA.")
else:
    df = dataframes[main_record_set_id]
    # Remove rows where numeric_field is missing or not numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if not pd.isna(df[numeric_field_id].mean()) else 0
    filtered = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered {len(filtered)} rows with {numeric_field_id} > {threshold:.2f}")
    # Normalize
    mean = filtered[numeric_field_id].mean()
    std = filtered[numeric_field_id].std()
    filtered[f"{numeric_field_id}_normalized"] = (filtered[numeric_field_id] - mean) / std
    print(filtered[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group field if present
    if group_field_id and group_field_id in filtered.columns:
        grouped = filtered.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nMean of {numeric_field_id} grouped by {group_field_id}:")
        print(grouped.head())

## 5. Visualization
Visualize the numeric field distribution and (if available) the grouping across the group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    df = dataframes[main_record_set_id]
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we used `mlcroissant` to automatically load and analyze the FAIR^2 mass spectrometry dataset, referencing all entities and fields by their Croissant `@id` identifiers. You can further explore more record sets and perform advanced processing as needed. The notebook demonstrates best practices for schema-driven, machine-actionable FAIR data handling in Python.